# Heart Failure Patient 30-Day Readmission Prediction

**Author:** Soham Garad  
**Field:** Artificial Intelligence & Machine Learning

## Overview
This project applies Machine Learning to clinical and demographic data from heart-failure patients to predict whether a patient is **readmitted within 30 days**.

> **Important:** In this dataset, the target column is `Readmitted_30_Days`. Therefore, the model predicts **30-day readmission**, not the initial diagnosis of heart failure.

## Objectives
- Explore and preprocess patient data.
- Analyze the target distribution.
- Encode categorical variables and scale numerical variables.
- Train and compare Logistic Regression, K-Nearest Neighbors (KNN), and Decision Tree models.
- Evaluate models using Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC.
- Identify the best-performing model.

## Disclaimer
This project is for educational and research purposes only. It is not a medical diagnostic tool and must not be used for clinical decision-making.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# 2. Load dataset
# Upload the CSV to the same folder as this notebook before running.
file_path = "dataset_12000_records (1)(2).csv"
data = pd.read_csv(file_path)

print("Dataset shape:", data.shape)
display(data.head())


In [ ]:
# 3. Basic dataset inspection
print("Columns:")
print(data.columns.tolist())

print("\nData types:")
display(data.dtypes)

print("\nMissing values:")
display(data.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", data.duplicated().sum())


In [ ]:
# 4. Target distribution
target = "Readmitted_30_Days"

print(data[target].value_counts())
print("\nProportion:")
print(data[target].value_counts(normalize=True))

plt.figure(figsize=(6,4))
sns.countplot(data=data, x=target)
plt.title("Target Distribution: 30-Day Readmission")
plt.xlabel("Readmitted within 30 Days (0 = No, 1 = Yes)")
plt.ylabel("Number of Patients")
plt.show()


### Observation
The dataset contains 12,000 patient records. The target is moderately imbalanced: approximately 70% are class 0 and 30% are class 1. Because of this imbalance, model evaluation should consider precision, recall, F1-score and ROC-AUC in addition to accuracy.


In [ ]:
# 5. Prepare features and target
# Patient_ID is an identifier, so it is excluded from model training.
X = data.drop(columns=[target, "Patient_ID"])
y = data[target]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))


In [ ]:
# 6. Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# 7. Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(4))


In [ ]:
# 8. Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, pipe) in zip(axes, trained_models.items()):
    pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# 9. ROC curves
plt.figure(figsize=(8,6))

for name, pipe in trained_models.items():
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 10. Train vs Test Accuracy
train_test_rows = []

for name, pipe in trained_models.items():
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    train_test_rows.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred)
    })

train_test_df = pd.DataFrame(train_test_rows)
display(train_test_df.round(4))

plot_df = train_test_df.set_index("Model")
plot_df.plot(kind="bar", figsize=(8,5))
plt.title("Train vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# 11. Detailed classification report for the best model
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

best_pred = best_model.predict(X_test)

print("Best model:", best_model_name)
print()
print(classification_report(y_test, best_pred, target_names=["No Readmission", "Readmission"]))


## Conclusion

The three classification algorithms are compared using multiple evaluation metrics. The model with the strongest ROC-AUC and overall test performance can be selected as the preferred model for this educational project.

The final model should be interpreted as a machine-learning demonstration for **30-day readmission prediction among heart-failure patients**, not as a clinical diagnostic system.
